# Notebook 07: Multi-Output LSTM Architecture Development for Joint SOH and RUL Prediction

**Project:** Machine Learning Advances in Lithium-Ion Battery Degradation and Lifetime Prediction  
**Dataset:** NASA Ames Prognostics Center of Excellence (PCoE) Battery Dataset (38 batteries, 2,881 cycles)  
**Model Framework:** PyTorch Multi-Output Recurrent Neural Network (Shared LSTM with Dual Regression Heads)

---

## Executive Summary & Modeling Objective
The objective of this stage is to build, train, validate, and evaluate **ONE Unified Multi-Output LSTM Model** that simultaneously predicts:
1. **State of Health (SOH %):** Battery capacity retention relative to the scientifically justified reference capacity.
2. **Remaining Useful Life (RUL cycles):** True remaining physical discharge cycles until the battery reaches the 80% End of Life (EOL) degradation threshold.

### Strict Methodological Rigor & Anti-Leakage Constraints:
1. **Shared Recurrent Core:** Two stacked standard LSTM layers (128 and 64 hidden units) capture joint temporal degradation dynamics from consecutive discharge cycles.
2. **Feature Scaling (Strict Zero-Leakage):** Standard scaling is fitted **strictly on training cycles only** and applied uniformly across sequence time steps. Validation and test sets are never fitted.
3. **Loss Masking for Right-Censored Data:** Missing RUL values for right-censored batteries (which did not reach 80% EOL) are handled using binary loss masks, avoiding arbitrary zero-imputation or target distortion.
4. **Adaptive Multi-Task Loss Balancing:** Inverse-variance weighting balances the numerical magnitudes between SOH percentage loss and RUL cycle loss.
5. **Original Physical Units:** All test metrics (MAE, MSE, RMSE, $R^2$) are reported in physical units (% for SOH, discharge cycles for RUL).
6. **Transfer Learning Readiness:** Model weights, configuration metadata, and feature scalers are serialized for immediate transfer to other battery datasets.


## Section 1: Environment Setup, Configuration, and Library Imports

We set up deterministic seeds across Python, NumPy, and PyTorch, configure the computation device (NVIDIA CUDA GPU if available), and establish output directories.


In [2]:
import os

# Ensure working directory is always the project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
import json
import random
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
os.environ['MPLCONFIGDIR'] = '.mpl_cache'

# 1. Deterministic Seeding for Scientific Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# 2. Device Configuration (CUDA GPU vs CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("=" * 70)
print(f"PyTorch Version   : {torch.__version__}")
print(f"Computation Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model         : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available    : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
print("=" * 70)

# 3. Output Directories
PREP_DIR   = 'data/intermediate/lstm_prepared_data'
MODELS_DIR = 'models/nasa_lstm'
VIZ_DIR    = os.path.join(MODELS_DIR, 'visualizations')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(VIZ_DIR, exist_ok=True)
print(f"Artifacts will be saved to: {os.path.abspath(MODELS_DIR)}")


PyTorch Version   : 2.13.0+cu130
Computation Device: cuda
GPU Model         : NVIDIA GeForce RTX 4060 Laptop GPU
VRAM Available    : 8.00 GB
Artifacts will be saved to: D:\deeplearning\project ml\models/nasa_lstm


C:\Users\CHEMBETI INDU KUMAR\.matplotlib is not a writable directory
Matplotlib created a temporary cache directory at C:\Users\CHEMBE~1\AppData\Local\Temp\matplotlib-zzjy_5a1 because there was an issue with the default path (C:\Users\CHEMBETI INDU KUMAR\.matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


## Section 2: Loading Existing Prepared Tensors and Metadata

We load the serialized PyTorch tensors and configuration metadata generated during Notebook 06.


In [4]:
# 1. Load Preparation Configuration
config_path = os.path.join(PREP_DIR, 'preparation_config.json')
with open(config_path, 'r') as f:
    prep_config = json.load(f)

print("=== Loaded Preparation Configuration ===")
print(f"Sequence Length (W)    : {prep_config['sequence_length']}")
print(f"Number of Features (D) : {prep_config['number_of_features']}")
print(f"Input Features List    : {prep_config['feature_names']}")
print(f"Targets List           : {prep_config['target_names']}")
print(f"Train Batteries ({len(prep_config['train_batteries'])}) : {prep_config['train_batteries'][:6]} ...")
print(f"Val Batteries   ({len(prep_config['validation_batteries'])}) : {prep_config['validation_batteries']}")
print(f"Test Batteries  ({len(prep_config['test_batteries'])}) : {prep_config['test_batteries']}")

# 2. Load PyTorch Tensors Packages
train_package = torch.load(os.path.join(PREP_DIR, 'train_tensors.pt'), weights_only=False)
val_package   = torch.load(os.path.join(PREP_DIR, 'val_tensors.pt'), weights_only=False)
test_package  = torch.load(os.path.join(PREP_DIR, 'test_tensors.pt'), weights_only=False)

# 3. Load Sequence Metadata
train_meta = pd.read_csv(os.path.join(PREP_DIR, 'train_sequences_metadata.csv'))
val_meta   = pd.read_csv(os.path.join(PREP_DIR, 'val_sequences_metadata.csv'))
test_meta  = pd.read_csv(os.path.join(PREP_DIR, 'test_sequences_metadata.csv'))

print(f"Loaded train sequences metadata: {len(train_meta)} rows")
print(f"Loaded val sequences metadata  : {len(val_meta)} rows")
print(f"Loaded test sequences metadata : {len(test_meta)} rows")


=== Loaded Preparation Configuration ===
Sequence Length (W)    : 5
Number of Features (D) : 13
Input Features List    : ['ambient_temperature', 'discharge_duration_s', 'voltage_measured_mean', 'voltage_measured_min', 'voltage_measured_max', 'voltage_measured_std', 'current_measured_mean', 'current_measured_min', 'temperature_measured_mean', 'temperature_measured_max', 'temperature_measured_min', 'voltage_load_mean', 'current_load_mean']
Targets List           : ['SOH_percent', 'RUL_cycles']
Train Batteries (26) : ['Battery_01', 'Battery_02', 'Battery_03', 'Battery_05', 'Battery_07', 'Battery_08'] ...
Val Batteries   (6) : ['Battery_06', 'Battery_12', 'Battery_15', 'Battery_21', 'Battery_25', 'Battery_27']
Test Batteries  (6) : ['Battery_04', 'Battery_16', 'Battery_20', 'Battery_24', 'Battery_32', 'Battery_38']
Loaded train sequences metadata: 1996 rows
Loaded val sequences metadata  : 303 rows
Loaded test sequences metadata : 430 rows


## Section 3: Rigorous Tensor and Metadata Verification

We verify tensor dimensions, data types, missing value absence, target ranges, and complete battery isolation prior to feature scaling.


In [6]:
# Verification Checks
verification_audit = []

for split_name, pkg in [('Train', train_package), ('Validation', val_package), ('Test', test_package)]:
    X_t = pkg['X']
    y_soh = pkg['y_soh']
    y_rul = pkg['y_rul']
    m_soh = pkg['mask_soh']
    m_rul = pkg['mask_rul']
    
    assert X_t.dim() == 3, f"{split_name} X must be 3D (N, W, D)"
    assert X_t.shape[1] == 5, f"{split_name} Sequence length must be 5"
    assert X_t.shape[2] == 13, f"{split_name} Feature count must be 13"
    assert torch.isnan(X_t).sum() == 0, f"{split_name} X contains NaNs!"
    
    valid_soh = y_soh[m_soh == 1.0]
    valid_rul = y_rul[m_rul == 1.0]
    
    verification_audit.append({
        'Split'          : split_name,
        'X_Shape'        : str(list(X_t.shape)),
        'y_SOH_Shape'    : str(list(y_soh.shape)),
        'y_RUL_Shape'    : str(list(y_rul.shape)),
        'Valid_SOH_Count': int(m_soh.sum().item()),
        'Valid_RUL_Count': int(m_rul.sum().item()),
        'SOH_Min_%'      : round(float(valid_soh.min()), 2),
        'SOH_Max_%'      : round(float(valid_soh.max()), 2),
        'RUL_Min_Cycles' : round(float(valid_rul.min()), 2),
        'RUL_Max_Cycles' : round(float(valid_rul.max()), 2),
        'X_NaN_Count'    : int(torch.isnan(X_t).sum())
    })

verif_df = pd.DataFrame(verification_audit)
print("=== Tensor Integrity Verification Summary ===")
print(verif_df.to_string(index=False))

# Assert Battery-Wise Separation Across Partitions
train_bats = set(prep_config['train_batteries'])
val_bats   = set(prep_config['validation_batteries'])
test_bats  = set(prep_config['test_batteries'])

assert len(train_bats & val_bats) == 0, "ERROR: Overlap between Train and Validation batteries!"
assert len(train_bats & test_bats) == 0, "ERROR: Overlap between Train and Test batteries!"
assert len(val_bats & test_bats) == 0, "ERROR: Overlap between Validation and Test batteries!"

print()
print("[VERIFICATION PASSED] All 3 partitions are strictly disjoint and verified leak-free.")


=== Tensor Integrity Verification Summary ===
     Split       X_Shape y_SOH_Shape y_RUL_Shape  Valid_SOH_Count  Valid_RUL_Count  SOH_Min_%  SOH_Max_%  RUL_Min_Cycles  RUL_Max_Cycles  X_NaN_Count
     Train [1996, 5, 13]   [1996, 1]   [1996, 1]             1947             1559       3.27     243.92             0.0           119.0            0
Validation  [303, 5, 13]    [303, 1]    [303, 1]              300              243       3.30     108.84             0.0            37.0            0
      Test  [430, 5, 13]    [430, 1]    [430, 1]              427              351       3.60     162.02             0.0            70.0            0

[VERIFICATION PASSED] All 3 partitions are strictly disjoint and verified leak-free.


## Section 4: Feature Scaling (Strict Training-Only Fit)

In compliance with our anti-leakage guidelines:
- We fit `StandardScaler` **exclusively on the training dataset** across all time steps.
- The fitted parameters ($\mu, \sigma$) are then applied to transform validation and test sequences.
- Validation and test data are never seen during fitting.
- The fitted scaler is serialized using Joblib for model deployment and transfer learning.


In [8]:
# Extract raw numpy arrays
X_train_raw = train_package['X'].numpy()
X_val_raw   = val_package['X'].numpy()
X_test_raw  = test_package['X'].numpy()

N_train, W, D = X_train_raw.shape
N_val         = X_val_raw.shape[0]
N_test        = X_test_raw.shape[0]

# 1. Fit StandardScaler strictly on Training Data (across all time steps)
scaler = StandardScaler()
X_train_2d = X_train_raw.reshape(-1, D)
scaler.fit(X_train_2d)

print("=== Fitted Feature Scaler Parameters (from Training Data Only) ===")
scaler_params = pd.DataFrame({
    'Feature_Index': range(1, D + 1),
    'Feature_Name' : prep_config['feature_names'],
    'Mean_mu'      : scaler.mean_,
    'Std_sigma'    : scaler.scale_
})
print(scaler_params.to_string(index=False))

# 2. Transform Train, Val, and Test Sequences
X_train_scaled = scaler.transform(X_train_2d).reshape(N_train, W, D)
X_val_scaled   = scaler.transform(X_val_raw.reshape(-1, D)).reshape(N_val, W, D)
X_test_scaled  = scaler.transform(X_test_raw.reshape(-1, D)).reshape(N_test, W, D)

# Convert to PyTorch float32 Tensors
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
X_val_t   = torch.tensor(X_val_scaled, dtype=torch.float32)
X_test_t  = torch.tensor(X_test_scaled, dtype=torch.float32)

y_soh_train_t = train_package['y_soh']
y_rul_train_t = train_package['y_rul']
m_soh_train_t = train_package['mask_soh']
m_rul_train_t = train_package['mask_rul']

y_soh_val_t = val_package['y_soh']
y_rul_val_t = val_package['y_rul']
m_soh_val_t = val_package['mask_soh']
m_rul_val_t = val_package['mask_rul']

y_soh_test_t = test_package['y_soh']
y_rul_test_t = test_package['y_rul']
m_soh_test_t = test_package['mask_soh']
m_rul_test_t = test_package['mask_rul']

# 3. Verify Scaled Distributions
print()
print("=== Post-Scaling Tensor Statistics ===")
print(f"Train Scaled X: mean = {X_train_t.mean():.4f} (expected ~0.0), std = {X_train_t.std():.4f} (expected ~1.0)")
print(f"Val   Scaled X: mean = {X_val_t.mean():.4f}, std = {X_val_t.std():.4f}")
print(f"Test  Scaled X: mean = {X_test_t.mean():.4f}, std = {X_test_t.std():.4f}")

# 4. Save Fitted Scaler for Transfer Learning
scaler_save_path = os.path.join(MODELS_DIR, 'feature_scaler.pkl')
joblib.dump(scaler, scaler_save_path)
print(f"Saved fitted StandardScaler to: {scaler_save_path}")


=== Fitted Feature Scaler Parameters (from Training Data Only) ===
 Feature_Index              Feature_Name     Mean_mu   Std_sigma
             1       ambient_temperature   18.694389   10.826598
             2      discharge_duration_s 3354.884314 1324.561574
             3     voltage_measured_mean    3.435734    0.168127
             4      voltage_measured_min    2.303205    0.316390
             5      voltage_measured_max    4.188055    0.022940
             6      voltage_measured_std    0.302277    0.079289
             7     current_measured_mean   -1.663463    0.620429
             8      current_measured_min   -2.548897    1.104315
             9 temperature_measured_mean   27.561228   12.732405
            10  temperature_measured_max   35.709893   15.707366
            11  temperature_measured_min   19.575504   10.653308
            12         voltage_load_mean    1.809394    0.868039
            13         current_load_mean    1.490662    0.953171

=== Post-Scaling Tenso

## Section 5: Defining the Multi-Output LSTM Architecture

We implement a shared recurrent core consisting of two unidirectional LSTM layers followed by dropout, a shared fully connected feature transformation layer, and two independent regression branches for joint SOH and RUL estimation.

```text
Input Sequence: (Batch Size, Sequence Length = 5, Features = 13)
      │
      ▼
LSTM Layer 1: 128 hidden units (returns all sequences)
      │
      ▼
LSTM Layer 2: 64 hidden units (returns last sequence timestep)
      │
      ▼
Dropout Layer: p = 0.2
      │
      ▼
Shared Dense Layer: 64 -> 32 units (ReLU activation)
      │
      ├───────────────────────────────┐
      │                               │
      ▼                               ▼
SOH Output Branch (Linear: 32 -> 1)   RUL Output Branch (Linear: 32 -> 1)
(State of Health Prediction, %)       (Remaining Useful Life Prediction, cycles)
```


In [10]:
class MultiOutputLSTM(nn.Module):
    # Shared Multi-Output LSTM Regression Network for joint SOH and RUL prediction.
    def __init__(self, input_size=13, hidden_dim1=128, hidden_dim2=64, dense_dim=32, dropout=0.2):
        super(MultiOutputLSTM, self).__init__()
        self.input_size  = input_size
        self.hidden_dim1 = hidden_dim1
        self.hidden_dim2 = hidden_dim2
        self.dense_dim   = dense_dim
        self.dropout_p   = dropout
        
        # 1. Stacked Standard LSTM Core
        self.lstm1 = nn.LSTM(input_size=input_size, hidden_size=hidden_dim1, batch_first=True)
        self.lstm2 = nn.LSTM(input_size=hidden_dim1, hidden_size=hidden_dim2, batch_first=True)
        
        # 2. Regularization & Shared Representation
        self.dropout = nn.Dropout(p=dropout)
        self.shared_dense = nn.Linear(hidden_dim2, dense_dim)
        self.relu = nn.ReLU()
        
        # 3. Dedicated Task Output Branches
        self.soh_head = nn.Linear(dense_dim, 1)
        self.rul_head = nn.Linear(dense_dim, 1)
        
    def forward(self, x):
        out1, _ = self.lstm1(x)               # (batch_size, 5, 128)
        out2, _ = self.lstm2(out1)             # (batch_size, 5, 64)
        
        last_timestep = out2[:, -1, :]        # (batch_size, 64)
        
        dropped = self.dropout(last_timestep) # (batch_size, 64)
        shared  = self.relu(self.shared_dense(dropped)) # (batch_size, 32)
        
        soh_pred = self.soh_head(shared)      # (batch_size, 1)
        rul_pred = self.rul_head(shared)      # (batch_size, 1)
        
        return soh_pred, rul_pred

# Instantiate model
model = MultiOutputLSTM(input_size=13, hidden_dim1=128, hidden_dim2=64, dense_dim=32, dropout=0.2).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("=== Multi-Output LSTM Model Summary ===")
print(model)
print(f"Total Trainable Parameters: {total_params:,}")


=== Multi-Output LSTM Model Summary ===
MultiOutputLSTM(
  (lstm1): LSTM(13, 128, batch_first=True)
  (lstm2): LSTM(128, 64, batch_first=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (shared_dense): Linear(in_features=64, out_features=32, bias=True)
  (relu): ReLU()
  (soh_head): Linear(in_features=32, out_features=1, bias=True)
  (rul_head): Linear(in_features=32, out_features=1, bias=True)
)
Total Trainable Parameters: 125,026


## Section 6: Loss Function, Multi-Task Weighting, and Optimizer Configuration

### 1. Target Loss Investigation & Adaptive Multi-Task Weighting
Because SOH (%) and RUL (cycles) possess different natural numerical scales:
- $\text{Var}(\text{SOH}_{\text{train}}) \approx 1110.86$
- $\text{Var}(\text{RUL}_{\text{train}}) \approx 523.68$

We use an adaptive **inverse-variance normalized multi-task loss**:
$$\mathcal{L}_{\text{total}} = w_{\text{SOH}} \cdot \mathcal{L}_{\text{SOH}} + w_{\text{RUL}} \cdot \mathcal{L}_{\text{RUL}}$$
where $w_{\text{SOH}} = \frac{1}{\sigma^2_{\text{SOH}}}$ and $w_{\text{RUL}} = \frac{1}{\sigma^2_{\text{RUL}}}$. This balances both gradients evenly during backpropagation.

### 2. Masked Loss Calculation
For right-censored samples where RUL is unknown, `mask_rul = 0`, ensuring gradients are only computed on observed targets:
$$\mathcal{L}_{\text{SOH}} = \frac{\sum_{i=1}^B m_{\text{SOH}, i} (y_{\text{SOH}, i} - \hat{y}_{\text{SOH}, i})^2}{\sum_{i=1}^B m_{\text{SOH}, i} + \epsilon}$$
$$\mathcal{L}_{\text{RUL}} = \frac{\sum_{i=1}^B m_{\text{RUL}, i} (y_{\text{RUL}, i} - \hat{y}_{\text{RUL}, i})^2}{\sum_{i=1}^B m_{\text{RUL}, i} + \epsilon}$$


In [12]:
# Compute Training Target Variances
valid_soh_train = y_soh_train_t[m_soh_train_t == 1.0]
valid_rul_train = y_rul_train_t[m_rul_train_t == 1.0]

var_soh = torch.var(valid_soh_train).item()
var_rul = torch.var(valid_rul_train).item()

weight_soh = 1.0 / var_soh
weight_rul = 1.0 / var_rul

print(f"Training SOH Variance (sigma^2): {var_soh:.2f} -> Normalizing Weight w_soh: {weight_soh:.6f}")
print(f"Training RUL Variance (sigma^2): {var_rul:.2f} -> Normalizing Weight w_rul: {weight_rul:.6f}")

class MaskedMultiTaskMSELoss(nn.Module):
    def __init__(self, w_soh=weight_soh, w_rul=weight_rul):
        super(MaskedMultiTaskMSELoss, self).__init__()
        self.w_soh = w_soh
        self.w_rul = w_rul
        
    def forward(self, pred_soh, pred_rul, y_soh, y_rul, mask_soh, mask_rul):
        diff_soh = (pred_soh - y_soh) ** 2
        loss_soh = (diff_soh * mask_soh).sum() / (mask_soh.sum() + 1e-8)
        
        diff_rul = (pred_rul - y_rul) ** 2
        loss_rul = (diff_rul * mask_rul).sum() / (mask_rul.sum() + 1e-8)
        
        total_loss = self.w_soh * loss_soh + self.w_rul * loss_rul
        return total_loss, loss_soh, loss_rul

criterion = MaskedMultiTaskMSELoss(w_soh=weight_soh, w_rul=weight_rul)

LEARNING_RATE = 0.001
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-5)

print(f"Optimizer: Adam (lr = {LEARNING_RATE}, weight_decay = 1e-5)")
print("Scheduler: ReduceLROnPlateau (factor = 0.5, patience = 5)")


Training SOH Variance (sigma^2): 1110.86 -> Normalizing Weight w_soh: 0.000900
Training RUL Variance (sigma^2): 523.68 -> Normalizing Weight w_rul: 0.001910
Optimizer: Adam (lr = 0.001, weight_decay = 1e-5)
Scheduler: ReduceLROnPlateau (factor = 0.5, patience = 5)


## Section 7: Model Training, Validation, and Early Stopping

We train the model for up to 100 epochs using mini-batches of size 32. At each epoch:
- Validation loss is calculated across all validation sequences.
- Model weights are check-pointed whenever validation loss achieves a new minimum.
- Early stopping halts training if validation loss fails to improve for 15 consecutive epochs.


In [14]:
# Create DataLoaders
BATCH_SIZE = 32
train_dataset = TensorDataset(X_train_t, y_soh_train_t, y_rul_train_t, m_soh_train_t, m_rul_train_t)
val_dataset   = TensorDataset(X_val_t, y_soh_val_t, y_rul_val_t, m_soh_val_t, m_rul_val_t)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader    = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

MAX_EPOCHS       = 100
PATIENCE         = 15
BEST_MODEL_PATH  = os.path.join(MODELS_DIR, 'best_multi_output_lstm.pth')

best_val_loss = float('inf')
patience_counter = 0
history = []

print("=== Starting Multi-Output LSTM Training ===")
print(f"Max Epochs: {MAX_EPOCHS} | Batch Size: {BATCH_SIZE} | Early Stopping Patience: {PATIENCE}")
print()

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    running_tot_loss, running_soh_loss, running_rul_loss = 0.0, 0.0, 0.0
    
    for bx, by_soh, by_rul, bm_soh, bm_rul in train_loader:
        bx = bx.to(device)
        by_soh, by_rul = by_soh.to(device), by_rul.to(device)
        bm_soh, bm_rul = bm_soh.to(device), bm_rul.to(device)
        
        optimizer.zero_grad()
        p_soh, p_rul = model(bx)
        loss, l_soh, l_rul = criterion(p_soh, p_rul, by_soh, by_rul, bm_soh, bm_rul)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        n_b = len(bx)
        running_tot_loss += loss.item() * n_b
        running_soh_loss += l_soh.item() * n_b
        running_rul_loss += l_rul.item() * n_b
        
    train_loss = running_tot_loss / len(train_dataset)
    train_soh  = running_soh_loss / len(train_dataset)
    train_rul  = running_rul_loss / len(train_dataset)
    
    model.eval()
    val_tot_loss, val_soh_loss, val_rul_loss = 0.0, 0.0, 0.0
    
    with torch.no_grad():
        for vx, vy_soh, vy_rul, vm_soh, vm_rul in val_loader:
            vx = vx.to(device)
            vy_soh, vy_rul = vy_soh.to(device), vy_rul.to(device)
            vm_soh, vm_rul = vm_soh.to(device), vm_rul.to(device)
            
            vp_soh, vp_rul = model(vx)
            v_loss, vl_soh, vl_rul = criterion(vp_soh, vp_rul, vy_soh, vy_rul, vm_soh, vm_rul)
            
            n_vb = len(vx)
            val_tot_loss += v_loss.item() * n_vb
            val_soh_loss += vl_soh.item() * n_vb
            val_rul_loss += vl_rul.item() * n_vb
            
    val_loss = val_tot_loss / len(val_dataset)
    val_soh  = val_soh_loss / len(val_dataset)
    val_rul  = val_rul_loss / len(val_dataset)
    
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step(val_loss)
    
    history.append({
        'epoch'          : epoch,
        'train_loss'     : train_loss,
        'val_loss'       : val_loss,
        'train_soh_mse'  : train_soh,
        'val_soh_mse'    : val_soh,
        'train_rul_mse'  : train_rul,
        'val_rul_mse'    : val_rul,
        'learning_rate'  : current_lr
    })
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save({
            'epoch'               : epoch,
            'model_state_dict'    : model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_loss'       : best_val_loss,
            'val_soh_mse'         : val_soh,
            'val_rul_mse'         : val_rul,
            'w_soh'               : weight_soh,
            'w_rul'               : weight_rul
        }, BEST_MODEL_PATH)
        saved_mark = " [* BEST CHECKPOINT SAVED]"
    else:
        patience_counter += 1
        saved_mark = ""
        
    if epoch % 5 == 0 or epoch == 1 or saved_mark != "":
        print(f"Epoch {epoch:3d}/{MAX_EPOCHS:3d} | Train: {train_loss:.4f} (SOH: {train_soh:.2f}, RUL: {train_rul:.2f}) | Val: {val_loss:.4f} (SOH: {val_soh:.2f}, RUL: {val_rul:.2f}) | lr: {current_lr:.1e}{saved_mark}")
        
    if patience_counter >= PATIENCE:
        print()
        print(f"Early stopping triggered at Epoch {epoch}! No validation improvement for {PATIENCE} epochs.")
        break

history_df = pd.DataFrame(history)
history_df.to_csv(os.path.join(MODELS_DIR, 'training_history.csv'), index=False)
print()
print(f"Training complete! Best Validation Loss: {best_val_loss:.4f}")
print(f"Saved best model weights to: {BEST_MODEL_PATH}")


=== Starting Multi-Output LSTM Training ===
Max Epochs: 100 | Batch Size: 32 | Early Stopping Patience: 15

Epoch   1/100 | Train: 9.2819 (SOH: 9101.90, RUL: 569.92) | Val: 4.2919 (SOH: 4620.84, RUL: 69.24) | lr: 1.0e-03 [* BEST CHECKPOINT SAVED]
Epoch   2/100 | Train: 6.4168 (SOH: 6019.71, RUL: 522.55) | Val: 2.2921 (SOH: 2306.56, RUL: 112.97) | lr: 1.0e-03 [* BEST CHECKPOINT SAVED]
Epoch   3/100 | Train: 3.6900 (SOH: 2999.49, RUL: 518.34) | Val: 0.7257 (SOH: 633.68, RUL: 81.33) | lr: 1.0e-03 [* BEST CHECKPOINT SAVED]
Epoch   4/100 | Train: 2.0966 (SOH: 1210.67, RUL: 527.20) | Val: 0.5931 (SOH: 513.04, RUL: 68.72) | lr: 1.0e-03 [* BEST CHECKPOINT SAVED]
Epoch   5/100 | Train: 1.7695 (SOH: 871.79, RUL: 515.66) | Val: 0.4322 (SOH: 333.27, RUL: 69.22) | lr: 1.0e-03 [* BEST CHECKPOINT SAVED]
Epoch   6/100 | Train: 1.7099 (SOH: 830.71, RUL: 503.81) | Val: 0.2083 (SOH: 108.94, RUL: 57.71) | lr: 1.0e-03 [* BEST CHECKPOINT SAVED]
Epoch  10/100 | Train: 0.8198 (SOH: 222.89, RUL: 324.22) | Val:

## Section 8: Model Evaluation and Comprehensive Visualization

We load the checkpointed best model and perform unbiased evaluation on the **untouched test set** across both targets in their original physical units:
- **MAE (Mean Absolute Error):** Average magnitude of prediction errors.
- **MSE (Mean Squared Error):** Average squared residuals.
- **RMSE (Root Mean Squared Error):** Standard deviation of unexplained variance.
- **$R^2$ Score (Coefficient of Determination):** Proportion of variance captured by the model.

We also generate all 6 diagnostic visualizations required by the project specifications.


In [16]:
# 1. Load Checkpointed Best Model
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Loaded best checkpoint from Epoch {checkpoint['epoch']} with Val Loss = {checkpoint['best_val_loss']:.4f}")

# 2. Inference on Untouched Test Set
with torch.no_grad():
    test_x_dev = X_test_t.to(device)
    p_soh_test_t, p_rul_test_t = model(test_x_dev)
    
p_soh_test = p_soh_test_t.cpu().squeeze().numpy()
p_rul_test = p_rul_test_t.cpu().squeeze().numpy()

y_soh_test = y_soh_test_t.squeeze().numpy()
y_rul_test = y_rul_test_t.squeeze().numpy()
m_soh_test = m_soh_test_t.squeeze().numpy()
m_rul_test = m_rul_test_t.squeeze().numpy()

# 3. Calculate Metrics on Valid Targets Only (Original Units)
soh_valid_idx = np.where(m_soh_test == 1.0)[0]
rul_valid_idx = np.where(m_rul_test == 1.0)[0]

soh_true = y_soh_test[soh_valid_idx]
soh_pred = p_soh_test[soh_valid_idx]

rul_true = y_rul_test[rul_valid_idx]
rul_pred = p_rul_test[rul_valid_idx]

def compute_regression_metrics(y_true, y_pred, target_name, unit):
    mae  = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_true, y_pred)
    return {
        'Target'       : target_name,
        'Unit'         : unit,
        'Samples_N'    : len(y_true),
        'MAE'          : round(float(mae), 4),
        'MSE'          : round(float(mse), 4),
        'RMSE'         : round(float(rmse), 4),
        'R2_Score'     : round(float(r2), 4)
    }

metrics_list = [
    compute_regression_metrics(soh_true, soh_pred, 'State of Health (SOH)', '%'),
    compute_regression_metrics(rul_true, rul_pred, 'Remaining Useful Life (RUL)', 'cycles')
]

test_metrics_df = pd.DataFrame(metrics_list)
print("=== Independent Test Set Performance Metrics (Original Physical Units) ===")
print(test_metrics_df.to_string(index=False))

test_metrics_df.to_csv(os.path.join(MODELS_DIR, 'test_evaluation_metrics.csv'), index=False)

# 4. Save Test Predictions DataFrame
test_preds_df = test_meta.copy()
test_preds_df['SOH_Predicted_%'] = p_soh_test
test_preds_df['RUL_Predicted_cycles'] = p_rul_test
test_preds_df['SOH_Residual'] = np.where(test_preds_df['Has_SOH'], test_preds_df['SOH_Target'] - test_preds_df['SOH_Predicted_%'], np.nan)
test_preds_df['RUL_Residual'] = np.where(test_preds_df['Has_RUL'], test_preds_df['RUL_Target'] - test_preds_df['RUL_Predicted_cycles'], np.nan)

test_preds_path = os.path.join(MODELS_DIR, 'test_predictions.csv')
test_preds_df.to_csv(test_preds_path, index=False)
print(f"Saved test predictions table to: {test_preds_path}")


Loaded best checkpoint from Epoch 30 with Val Loss = 0.0895
=== Independent Test Set Performance Metrics (Original Physical Units) ===
                     Target   Unit  Samples_N     MAE       MSE    RMSE  R2_Score
      State of Health (SOH)      %        427 28.2258 3028.1340 55.0285   -1.3312
Remaining Useful Life (RUL) cycles        351 12.6037  599.9948 24.4948   -0.9978
Saved test predictions table to: models/nasa_lstm\test_predictions.csv


In [17]:
# ── Plot 1: Training and Validation Loss Curves ──────────────────────────────
plt.figure(figsize=(10, 5))
plt.plot(history_df['epoch'], history_df['train_loss'], label='Train Total Loss (Weighted)', color='#2980b9', lw=2)
plt.plot(history_df['epoch'], history_df['val_loss'], label='Val Total Loss (Weighted)', color='#e74c3c', lw=2, linestyle='--')
best_ep = checkpoint['epoch']
plt.axvline(best_ep, color='#27ae60', linestyle=':', lw=2, label=f'Best Model (Epoch {best_ep})')
plt.title('Multi-Output LSTM: Training vs Validation Loss Curves', fontsize=12, fontweight='bold')
plt.xlabel('Epoch', fontsize=11)
plt.ylabel('Normalized Multi-Task Loss', fontsize=11)
plt.grid(alpha=0.3)
plt.legend(frameon=True)
p1 = os.path.join(VIZ_DIR, 'training_validation_loss.png')
plt.savefig(p1, dpi=120, bbox_inches='tight')
plt.close()
print(f"Saved: {p1}")

# ── Plot 2: Actual vs Predicted SOH ───────────────────────────────────────────
plt.figure(figsize=(7, 6))
plt.scatter(soh_true, soh_pred, alpha=0.6, color='#2980b9', edgecolor='k', s=35, label='Test Samples')
min_s, max_s = min(soh_true.min(), soh_pred.min()) - 2, max(soh_true.max(), soh_pred.max()) + 2
plt.plot([min_s, max_s], [min_s, max_s], color='#c0392b', lw=2, linestyle='--', label='Ideal Fit (y = x)')
r2_soh_val = test_metrics_df.loc[test_metrics_df['Target'].str.contains('SOH'), 'R2_Score'].values[0]
rmse_soh_val = test_metrics_df.loc[test_metrics_df['Target'].str.contains('SOH'), 'RMSE'].values[0]
plt.title(f'Actual vs Predicted SOH (Test Split)\nR^2 = {r2_soh_val:.4f} | RMSE = {rmse_soh_val:.2f}%', fontsize=12, fontweight='bold')
plt.xlabel('Actual SOH (%)', fontsize=11)
plt.ylabel('Predicted SOH (%)', fontsize=11)
plt.grid(alpha=0.3)
plt.legend()
p2 = os.path.join(VIZ_DIR, 'actual_vs_predicted_soh.png')
plt.savefig(p2, dpi=120, bbox_inches='tight')
plt.close()
print(f"Saved: {p2}")

# ── Plot 3: Actual vs Predicted RUL ───────────────────────────────────────────
plt.figure(figsize=(7, 6))
plt.scatter(rul_true, rul_pred, alpha=0.6, color='#8e44ad', edgecolor='k', s=35, label='Test Samples')
min_r, max_r = min(rul_true.min(), rul_pred.min()) - 2, max(rul_true.max(), rul_pred.max()) + 2
plt.plot([min_r, max_r], [min_r, max_r], color='#c0392b', lw=2, linestyle='--', label='Ideal Fit (y = x)')
r2_rul_val = test_metrics_df.loc[test_metrics_df['Target'].str.contains('RUL'), 'R2_Score'].values[0]
rmse_rul_val = test_metrics_df.loc[test_metrics_df['Target'].str.contains('RUL'), 'RMSE'].values[0]
plt.title(f'Actual vs Predicted RUL (Test Split)\nR^2 = {r2_rul_val:.4f} | RMSE = {rmse_rul_val:.2f} cycles', fontsize=12, fontweight='bold')
plt.xlabel('Actual RUL (Cycles)', fontsize=11)
plt.ylabel('Predicted RUL (Cycles)', fontsize=11)
plt.grid(alpha=0.3)
plt.legend()
p3 = os.path.join(VIZ_DIR, 'actual_vs_predicted_rul.png')
plt.savefig(p3, dpi=120, bbox_inches='tight')
plt.close()
print(f"Saved: {p3}")

# ── Plot 4: SOH Prediction Error Distribution ─────────────────────────────────
res_soh = soh_true - soh_pred
plt.figure(figsize=(8, 5))
sns.histplot(res_soh, kde=True, color='#2980b9', bins=30, edgecolor='black', alpha=0.7)
plt.axvline(0, color='red', linestyle='--', lw=2, label='Zero Error')
plt.axvline(res_soh.mean(), color='black', linestyle=':', lw=2, label=f'Mean Error: {res_soh.mean():.2f}%')
plt.title(f'SOH Residual Distribution (Actual - Predicted)\nMean = {res_soh.mean():.2f}% | Std = {res_soh.std():.2f}%', fontsize=12, fontweight='bold')
plt.xlabel('SOH Prediction Error (%)', fontsize=11)
plt.ylabel('Frequency', fontsize=11)
plt.grid(alpha=0.3)
plt.legend()
p4 = os.path.join(VIZ_DIR, 'soh_prediction_error_distribution.png')
plt.savefig(p4, dpi=120, bbox_inches='tight')
plt.close()
print(f"Saved: {p4}")

# ── Plot 5: RUL Prediction Error Distribution ─────────────────────────────────
res_rul = rul_true - rul_pred
plt.figure(figsize=(8, 5))
sns.histplot(res_rul, kde=True, color='#8e44ad', bins=30, edgecolor='black', alpha=0.7)
plt.axvline(0, color='red', linestyle='--', lw=2, label='Zero Error')
plt.axvline(res_rul.mean(), color='black', linestyle=':', lw=2, label=f'Mean Error: {res_rul.mean():.2f} cyc')
plt.title(f'RUL Residual Distribution (Actual - Predicted)\nMean = {res_rul.mean():.2f} cycles | Std = {res_rul.std():.2f} cycles', fontsize=12, fontweight='bold')
plt.xlabel('RUL Prediction Error (Cycles)', fontsize=11)
plt.ylabel('Frequency', fontsize=11)
plt.grid(alpha=0.3)
plt.legend()
p5 = os.path.join(VIZ_DIR, 'rul_prediction_error_distribution.png')
plt.savefig(p5, dpi=120, bbox_inches='tight')
plt.close()
print(f"Saved: {p5}")

# ── Plot 6: Battery-Wise Trajectory Analysis Across Test Batteries ─────────────
test_bats_list = prep_config['test_batteries']
n_tb = len(test_bats_list)

fig, axes = plt.subplots(n_tb, 2, figsize=(14, 3.2 * n_tb), sharex=False)

for idx, b_id in enumerate(test_bats_list):
    b_df = test_preds_df[test_preds_df['Battery_ID'] == b_id].sort_values('End_Cycle')
    
    ax_soh = axes[idx, 0]
    ax_soh.plot(b_df['End_Cycle'], b_df['SOH_Target'], label='Actual SOH', color='#2980b9', lw=2)
    ax_soh.plot(b_df['End_Cycle'], b_df['SOH_Predicted_%'], label='LSTM Predicted SOH', color='#e67e22', linestyle='--', lw=2)
    ax_soh.axhline(80.0, color='red', linestyle=':', label='80% EOL Threshold')
    ax_soh.set_title(f'{b_id}: SOH Degradation Tracking', fontsize=11, fontweight='bold')
    ax_soh.set_ylabel('SOH (%)', fontsize=10)
    ax_soh.grid(alpha=0.3)
    if idx == 0:
        ax_soh.legend(fontsize=9, loc='upper right')
        
    ax_rul = axes[idx, 1]
    if b_df['Has_RUL'].any():
        ax_rul.plot(b_df['End_Cycle'], b_df['RUL_Target'], label='Actual RUL', color='#8e44ad', lw=2)
        ax_rul.plot(b_df['End_Cycle'], b_df['RUL_Predicted_cycles'], label='LSTM Predicted RUL', color='#27ae60', linestyle='--', lw=2)
        ax_rul.set_title(f'{b_id}: RUL Countdown Trajectory', fontsize=11, fontweight='bold')
    else:
        ax_rul.plot(b_df['End_Cycle'], b_df['RUL_Predicted_cycles'], label='Predicted RUL (Unlabeled True)', color='#27ae60', linestyle='--', lw=2)
        ax_rul.set_title(f'{b_id}: RUL Predicted (Right-Censored Unlabeled Battery)', fontsize=11, fontweight='bold')
        
    ax_rul.set_ylabel('RUL (Cycles)', fontsize=10)
    ax_rul.grid(alpha=0.3)
    if idx == 0:
        ax_rul.legend(fontsize=9, loc='upper right')
        
    if idx == n_tb - 1:
        ax_soh.set_xlabel('Discharge Cycle Number', fontsize=11)
        ax_rul.set_xlabel('Discharge Cycle Number', fontsize=11)

plt.tight_layout()
p6 = os.path.join(VIZ_DIR, 'battery_wise_prediction_analysis.png')
plt.savefig(p6, dpi=120, bbox_inches='tight')
plt.close()
print(f"Saved: {p6}")


Saved: models/nasa_lstm\visualizations\training_validation_loss.png
Saved: models/nasa_lstm\visualizations\actual_vs_predicted_soh.png
Saved: models/nasa_lstm\visualizations\actual_vs_predicted_rul.png
Saved: models/nasa_lstm\visualizations\soh_prediction_error_distribution.png
Saved: models/nasa_lstm\visualizations\rul_prediction_error_distribution.png
Saved: models/nasa_lstm\visualizations\battery_wise_prediction_analysis.png


## Section 9: Saving the Best Model, Metadata, and Transfer Learning Export

We save the model architecture configuration JSON, verify reloading of the trained checkpoint, and run inference on a sample tensor to prove that the pipeline is self-contained and transfer-learning ready.


In [19]:
# 1. Export Master Architecture Configuration
arch_config = {
    'model_name'           : 'MultiOutputLSTM',
    'architecture'         : {
        'input_size'       : 13,
        'hidden_dim1'      : 128,
        'hidden_dim2'      : 64,
        'dense_dim'        : 32,
        'dropout'          : 0.2,
        'soh_output_dim'   : 1,
        'rul_output_dim'   : 1
    },
    'training_parameters'  : {
        'batch_size'       : BATCH_SIZE,
        'learning_rate'    : LEARNING_RATE,
        'loss_function'    : 'MaskedMultiTaskMSELoss',
        'w_soh'            : weight_soh,
        'w_rul'            : weight_rul,
        'best_epoch'       : checkpoint['epoch'],
        'best_val_loss'    : checkpoint['best_val_loss'],
        'random_seed'      : RANDOM_SEED
    },
    'features_order'       : prep_config['feature_names'],
    'targets'              : ['SOH_percent', 'RUL_cycles'],
    'sequence_length'      : prep_config['sequence_length'],
    'test_metrics'         : test_metrics_df.to_dict(orient='records'),
    'device_trained_on'    : str(device)
}

arch_config_path = os.path.join(MODELS_DIR, 'model_architecture_config.json')
with open(arch_config_path, 'w') as f:
    json.dump(arch_config, f, indent=2)
print(f"Saved model architecture metadata to: {arch_config_path}")

# 2. Transfer Learning Verification: Load model and scaler independently and infer
print()
print("=== Verifying Transfer Learning Readiness ===")
loaded_scaler = joblib.load(os.path.join(MODELS_DIR, 'feature_scaler.pkl'))
loaded_model = MultiOutputLSTM(
    input_size=arch_config['architecture']['input_size'],
    hidden_dim1=arch_config['architecture']['hidden_dim1'],
    hidden_dim2=arch_config['architecture']['hidden_dim2'],
    dense_dim=arch_config['architecture']['dense_dim'],
    dropout=arch_config['architecture']['dropout']
).to(device)

loaded_ckpt = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=False)
loaded_model.load_state_dict(loaded_ckpt['model_state_dict'])
loaded_model.eval()

# Dummy raw sequence (1, 5, 13) simulating an external battery dataset
dummy_raw_sample = np.random.uniform(low=2.0, high=4.0, size=(1, 5, 13))
dummy_scaled = loaded_scaler.transform(dummy_raw_sample.reshape(-1, 13)).reshape(1, 5, 13)
dummy_tensor = torch.tensor(dummy_scaled, dtype=torch.float32).to(device)

with torch.no_grad():
    infer_soh, infer_rul = loaded_model(dummy_tensor)

print(f"Sample Transfer Inference Output:")
print(f"  Predicted SOH: {infer_soh.item():.2f}%")
print(f"  Predicted RUL: {infer_rul.item():.2f} cycles")
print("[VERIFICATION SUCCESS] Model and scaler are fully portable and ready for Transfer Learning!")

print("=" * 75)
print("NOTEBOOK 07: MULTI-OUTPUT LSTM MODEL DEVELOPMENT COMPLETE")
print("=" * 75)


Saved model architecture metadata to: models/nasa_lstm\model_architecture_config.json

=== Verifying Transfer Learning Readiness ===
Sample Transfer Inference Output:
  Predicted SOH: 83.38%
  Predicted RUL: 1.65 cycles
[VERIFICATION SUCCESS] Model and scaler are fully portable and ready for Transfer Learning!
NOTEBOOK 07: MULTI-OUTPUT LSTM MODEL DEVELOPMENT COMPLETE
